In [1]:
import pandas as pd
import numpy as np
import re
import html
from collections import Counter

In [2]:
import pandas as pd

se_tags = {
    "java", "c#", "javascript", "c++", "python", "android", "ios", "sql",
    "git", "oop", "multithreading", "design-patterns", "architecture",
    "database-design", "unit-testing", "version-control", "api",
    "web-services", "debugging", "mvc"
}

net_tags = {
    "networking", "linux", "ubuntu", "security", "bash", "apache", "http",
    "ssh", "sockets", "dns", "ssl", "centos", "nginx", "tcp", "routing",
    "wireless-networking", "ftp", "proxy", "active-directory", "amazon-ec2"
}

ai_tags = {
    "opencv",
    "image-processing",
    "r",
    "matlab",
    "algorithm",
    "probability",
    "linear-algebra",
    "matrices",
    "vector",
    "data-structures"
}







In [3]:
se = pd.read_csv("train_se_clean.csv")
net = pd.read_csv("train_net_clean.csv")
ai = pd.read_csv("train_ai_clean.csv")

In [4]:
def merge_title_body(df):
    df = df.copy()
    df["Title"] = df["Title"].fillna("").astype(str)
    df["Body"] = df["Body"].fillna("").astype(str)

    df["text"] = (
        df["Title"].str.strip() + " " + df["Body"].str.strip()
    ).str.replace(r"\s+", " ", regex=True).str.strip()

    return df

In [5]:
se = merge_title_body(se)
net = merge_title_body(net)
ai = merge_title_body(ai)

In [6]:
def filter_tags(tags_text, allowed_tags):
    tags = str(tags_text).split()
    kept = [tag for tag in tags if tag in allowed_tags]
    return " ".join(kept)

se["Tags_filtered"] = se["Tags"].apply(lambda x: filter_tags(x, se_tags))
net["Tags_filtered"] = net["Tags"].apply(lambda x: filter_tags(x, net_tags))
ai["Tags_filtered"] = ai["Tags"].apply(lambda x: filter_tags(x, ai_tags))

# حذف الصفوف التي لم يبق فيها أي tag من التاغات المحددة
se = se[se["Tags_filtered"].str.strip() != ""].copy()
net = net[net["Tags_filtered"].str.strip() != ""].copy()
ai = ai[ai["Tags_filtered"].str.strip() != ""].copy()

# إذا بدك تستبدل Tags الأصلية بالمفلترة
se["Tags"] = se["Tags_filtered"]
net["Tags"] = net["Tags_filtered"]
ai["Tags"] = ai["Tags_filtered"]

# حذف العمود المؤقت
se.drop(columns=["Tags_filtered"], inplace=True)
net.drop(columns=["Tags_filtered"], inplace=True)
ai.drop(columns=["Tags_filtered"], inplace=True)

In [7]:
print("SE shape:", se.shape)
print("NET shape:", net.shape)
print("AI shape:", ai.shape)

print(se[["text", "Tags"]].head(3))
print(net[["text", "Tags"]].head(3))
print(ai[["text", "Tags"]].head(3))

SE shape: (2236180, 5)
NET shape: (400621, 5)
AI shape: (132782, 5)
                                                text Tags
0  How do I replace special characters in a URL? ...   c#
1  How to modify whois contact details? <pre><cod...  api
2  How to fetch an XML feed using asp.net <p>I've...   c#
                                                text                    Tags
0  setting proxy in active directory environment ...  proxy active-directory
1  High load on X3220 Quad Core Linux Apache serv...                   linux
2  is ssl secure on both ways? <p>I know that cer...                     ssl
                                                text              Tags
0  How to check if an uploaded file is an image w...  image-processing
1  R Error Invalid type (list) for variable <p>I ...          r matlab
2  Crappy Random Number Generator <p>This may sou...         algorithm


In [8]:
full_df = pd.concat([se, net, ai], ignore_index=True)
full_df = full_df[["text", "Tags"]]


In [9]:
print("Final shape:", full_df.shape)
print(full_df.head())


full_df.to_csv("train_merged_text_tags.csv", index=False)

Final shape: (2769583, 2)
                                                text        Tags
0  How do I replace special characters in a URL? ...          c#
1  How to modify whois contact details? <pre><cod...         api
2  How to fetch an XML feed using asp.net <p>I've...          c#
3  .NET library for generating javascript? <p>Do ...  javascript
4  SQL Server : procedure call, inline concatenat...         sql


In [3]:
import re
import html

NORMALIZATION_MAP = {
    "c sharp": "c#",
    "c-sharp": "c#",
    "c plus plus": "c++",
    "cpp": "c++",
    "js": "javascript",
    "nodejs": "node.js",
    "asp net": "asp.net",
    "ms sql": "sql server",
    "mssql": "sql server",
    "postgres": "postgresql",
}

def normalize_terms(text):
    for src, tgt in NORMALIZATION_MAP.items():
        pattern = r"\b" + re.escape(src) + r"\b"
        text = re.sub(pattern, tgt, text)
    return text

def clean_technical_text(text):
    text = str(text)

    # فك HTML
    text = html.unescape(text)
# code/pre blocks
    text = re.sub(r"<code>.*?</code>", " CODEBLOCK ", text, flags=re.DOTALL | re.IGNORECASE)
    text = re.sub(r"<pre>.*?</pre>", " CODEBLOCK ", text, flags=re.DOTALL | re.IGNORECASE)
# حذف HTML tags
    text = re.sub(r"<.*?>", " ", text)
    # lowercase
    text = text.lower()

    # normalization (مهم جداً)
    text = normalize_terms(text)

   # replace urls / emails
    text = re.sub(r"http\S+|www\S+", " URL ", text)
    text = re.sub(r"\b[\w\.-]+@[\w\.-]+\.\w+\b", " EMAIL ", text)
    

    # الحفاظ على الرموز التقنية
    text = re.sub(r"[^a-z0-9\s\#\+\.\-_/]", " ", text)

    # تنظيف المسافات
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [4]:

full_df = pd.read_csv("train_merged_text_tags.csv")
# 4) Keep raw version before cleaning
##full_df["text_raw"] = full_df["text"]


In [5]:
# apply preprocessing

full_df["text"] = full_df["text"].apply(clean_technical_text)

# remove empty rows after cleaning
full_df = full_df[
    (full_df["text"].str.strip() != "") &
    (full_df["Tags"].str.strip() != "")
].copy()
# check result
print("Final shape after preprocessing:", full_df.shape)
print(full_df.head())

# optional save
full_df.to_csv("merged_text_tags_clean.csv", index=False)

Final shape after preprocessing: (2769583, 2)
                                                text        Tags
0  how do i replace special characters in a url t...          c#
1  how to modify whois contact details codeblock ...         api
2  how to fetch an xml feed using asp.net i ve de...          c#
3  .net library for generating javascript do you ...  javascript
4  sql server procedure call inline concatenation...         sql


In [2]:
full_df = pd.read_csv("merged_text_tags_clean.csv")

In [ ]:
import pandas as pd

# تقسيم tags إلى list
full_df["Tags"] = full_df["Tags"].astype(str).apply(lambda x: x.split())

# دمج الصفوف المتكررة حسب text
df_merged = (
    full_df.groupby("text", as_index=False)["Tags"]
    .apply(lambda tag_lists: sorted(set(tag for tags in tag_lists for tag in tags)))
)

# إعادة Tags كنص مفصول بمسافة
df_merged["Tags"] = df_merged["Tags"].apply(lambda tags: " ".join(tags))

# حفظ النسخة المدموجة
df_merged.to_csv("merged_text_tags_grouped.csv", index=False)

# فحص
print("قبل الدمج:", full_df.shape)
print("بعد الدمج:", df_merged.shape)
print(df_merged.head(10))

قبل الدمج: (2769583, 2)
بعد الدمج: (1879950, 2)
                                                text        Tags
0  # + items .append is not a function codeblock ...  javascript
1  # - how to parallel code that lock several obj...          c#
2  # . what do and # do in this code codeblock tr...  javascript
3  # .dialog is not a function error after using ...  javascript
4  # .dialog is not a function error i am trying ...  javascript
5  # /bin/bash - no such file or directory i ve c...        bash
6  # /usr/bin/env interpreter arguments -- portab...       linux
7  # /usr/bin/env python getting command not foun...      python
8  # /usr/bin/env ruby is not found in cron i hav...        bash
9  # can not been used as separator of fields for...           r


In [7]:
df_merged["num_tags"] = df_merged["Tags"].str.split().apply(len)
print(df_merged["num_tags"].value_counts().sort_index())

num_tags
1    1645999
2     214068
3      18586
4       1246
5         51
Name: count, dtype: int64


In [3]:
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer

# 1) قراءة الملف بعد الدمج
df = pd.read_csv("merged_text_tags_grouped.csv")

In [21]:
df["text"] = df["text"].fillna("").astype(str).str.strip()
df["Tags"] = df["Tags"].fillna("").astype(str).str.strip()

df = df[(df["text"] != "") & (df["Tags"] != "")].copy()

# 3) تحويل Tags إلى قائمة
df["Tags"] = df["Tags"].apply(lambda x: x.split())
df=df.iloc[:700000].copy()
# 4) فصل X و y
X = df["text"]
y = df["Tags"]

In [22]:
# 5) تقسيم البيانات
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5,
    random_state=42
)

In [23]:
# 6) MultiLabelBinarizer
mlb = MultiLabelBinarizer()

y_train_bin = mlb.fit_transform(y_train)
y_val_bin = mlb.transform(y_val)
y_test_bin = mlb.transform(y_test)

# 7) حفظ الـ binarizer
joblib.dump(mlb, "multilabel_binarizer2.pkl")

['multilabel_binarizer2.pkl']

In [24]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multiclass import OneVsRestClassifier

# 1) TF-IDF
vectorizer = TfidfVectorizer(
    ngram_range=(1, 1),
    min_df=5,
    max_df=0.9,
    max_features=150000,
    sublinear_tf=True
)

# ⚠️ fit فقط على train
X_train_vec = vectorizer.fit_transform(X_train)

# transform للباقي
X_val_vec = vectorizer.transform(X_val)
X_test_vec = vectorizer.transform(X_test)

print("شكل X_train:", X_train_vec.shape)

شكل X_train: (400000, 44565)


In [25]:
import joblib

joblib.dump(vectorizer, "tfidf_vectorizer2.pkl")

['tfidf_vectorizer2.pkl']

In [34]:
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score



model = OneVsRestClassifier(
    LogisticRegression(
        solver="saga",
        max_iter=120,
        random_state=42,
        n_jobs=-1
    ),
    n_jobs=-1
)

model.fit(X_train_vec, y_train_bin)




,"estimator estimator: estimator objectA regressor or a classifier that implements :term:`fit`.When a classifier is passed, :term:`decision_function` will be usedin priority and it will fallback to :term:`predict_proba` if it is notavailable.When a regressor is passed, :term:`predict` is used.",LogisticRegre...solver='saga')
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation: the `n_classes`one-vs-rest problems are computed in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: 0.20 `n_jobs` default changed from 1 to None",-1
,"verbose verbose: int, default=0The verbosity level, if non zero, progress messages are printed.Below 50, the output is sent to stderr. Otherwise, the output is sentto stdout. The frequency of the messages increases with the verbositylevel, reporting all iterations at 10. See :class:`joblib.Parallel` formore details... versionadded:: 1.1",0
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=No

In [35]:
val_scores = model.predict_proba(X_val_vec)

# If returned as list of arrays, convert to (n_samples, n_labels)
if isinstance(val_scores, list):
    val_scores = np.array([p[:, 1] for p in val_scores]).T


In [36]:
def top_k_binary_predictions(y_scores, k):
    y_pred = np.zeros_like(y_scores, dtype=int)
    topk_idx = np.argsort(-y_scores, axis=1)[:, :k]

    for i in range(y_scores.shape[0]):
        y_pred[i, topk_idx[i]] = 1

    return y_pred

In [37]:
def evaluate_at_k(y_true, y_scores, k):
    y_pred_k = top_k_binary_predictions(y_scores, k)

    precision = precision_score(y_true, y_pred_k, average="micro", zero_division=0)
    recall = recall_score(y_true, y_pred_k, average="micro", zero_division=0)
    f1 = f1_score(y_true, y_pred_k, average="micro", zero_division=0)

    return precision, recall, f1

In [38]:
for k in [1, 2, 3,4]:
    p, r, f = evaluate_at_k(y_val_bin, val_scores, k)

    print(f"\n===== Top-{k} Metrics =====")
    print(f"Precision@{k}: {p:.4f}")
    print(f"Recall@{k}:    {r:.4f}")
    print(f"F1@{k}:        {f:.4f}")



===== Top-1 Metrics =====
Precision@1: 0.8398
Recall@1:    0.7390
F1@1:        0.7862

===== Top-2 Metrics =====
Precision@2: 0.5031
Recall@2:    0.8855
F1@2:        0.6417

===== Top-3 Metrics =====
Precision@3: 0.3533
Recall@3:    0.9327
F1@3:        0.5125

===== Top-4 Metrics =====
Precision@4: 0.2714
Recall@4:    0.9553
F1@4:        0.4227


In [39]:
joblib.dump(model, "best_logistic2.pkl")

['best_logistic2.pkl']

In [40]:

import numpy as np

test_scores = model.predict_proba(X_test_vec)

# إذا رجعت list، نحولها إلى array
if isinstance(test_scores, list):
    test_scores = np.array([p[:, 1] for p in test_scores]).T

In [41]:
for k in [1, 2, 3, 4]:
    p, r, f = evaluate_at_k(y_test_bin, test_scores, k)

    print(f"\n===== Top-{k} Test Metrics =====")
    print(f"Precision@{k}: {p:.4f}")
    print(f"Recall@{k}:    {r:.4f}")
    print(f"F1@{k}:        {f:.4f}")


===== Top-1 Test Metrics =====
Precision@1: 0.8400
Recall@1:    0.7386
F1@1:        0.7861

===== Top-2 Test Metrics =====
Precision@2: 0.5033
Recall@2:    0.8852
F1@2:        0.6418

===== Top-3 Test Metrics =====
Precision@3: 0.3535
Recall@3:    0.9327
F1@3:        0.5127

===== Top-4 Test Metrics =====
Precision@4: 0.2713
Recall@4:    0.9544
F1@4:        0.4226


In [6]:
df["text"] = df["text"].fillna("").astype(str).str.strip()
df["Tags"] = df["Tags"].fillna("").astype(str).str.strip()

df = df[(df["text"] != "") & (df["Tags"] != "")].copy()

# 3) تحويل Tags إلى قائمة
df["Tags"] = df["Tags"].apply(lambda x: x.split())
df=df.iloc[:500000].copy()
# 4) فصل X و y
X = df["text"]
y = df["Tags"]

In [7]:
# 5) تقسيم البيانات
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5,
    random_state=42
)

In [8]:
# 6) MultiLabelBinarizer
mlb = MultiLabelBinarizer()

y_train_bin = mlb.fit_transform(y_train)
y_val_bin = mlb.transform(y_val)
y_test_bin = mlb.transform(y_test)

# 7) حفظ الـ binarizer
joblib.dump(mlb, "multilabel_binarizer2.pkl")

['multilabel_binarizer2.pkl']

In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multiclass import OneVsRestClassifier

# 1) TF-IDF
vectorizer = TfidfVectorizer(
    ngram_range=(1, 1),
    min_df=5,
    max_df=0.9,
    max_features=100000,
    sublinear_tf=True
)

# ⚠️ fit فقط على train
X_train_vec = vectorizer.fit_transform(X_train)

# transform للباقي
X_val_vec = vectorizer.transform(X_val)
X_test_vec = vectorizer.transform(X_test)

print("شكل X_train:", X_train_vec.shape)

شكل X_train: (400000, 44565)


In [10]:
joblib.dump(vectorizer, "tfidf_vectorizersvc2.pkl")

['tfidf_vectorizersvc2.pkl']

In [11]:
import joblib
from sklearn.svm import LinearSVC
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import f1_score, jaccard_score

model = OneVsRestClassifier(
    LinearSVC(
        C=1,
        max_iter=7000,
        random_state=42
    ),
    n_jobs=-1
)

model.fit(X_train_vec, y_train_bin)



,"estimator estimator: estimator objectA regressor or a classifier that implements :term:`fit`.When a classifier is passed, :term:`decision_function` will be usedin priority and it will fallback to :term:`predict_proba` if it is notavailable.When a regressor is passed, :term:`predict` is used.",LinearSVC(C=1...ndom_state=42)
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation: the `n_classes`one-vs-rest problems are computed in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: 0.20 `n_jobs` default changed from 1 to None",-1
,"verbose verbose: int, default=0The verbosity level, if non zero, progress messages are printed.Below 50, the output is sent to stderr. Otherwise, the output is sentto stdout. The frequency of the messages increases with the verbositylevel, reporting all iterations at 10. See :class:`joblib.Parallel` formore details... versionadded:: 1.1",0
,"penalty penalty: {'l1', 'l2'}, default='l2'Specifies the norm used in the penalization. The 'l2'penalty is the standard used in SVC. The 'l1' leads to ``coef_``vectors that are sparse.",'l2'
,"loss loss: {'hinge', 'squared_hinge'}, default='squared_hinge'Specifies the loss function. 'hinge' is the standard SVM loss(used e.g. by the SVC class) while 'squared_hinge' is thesquare of the hinge loss. The combination of ``penalty='l1'``and ``loss='hinge'`` is not supported.",'squared_hinge'
,"dual dual: ""auto"" or bool, default=""auto""Select the algorithm to either solve the dual or primaloptimization problem. Prefer dual=False when n_samples > n_features.`dual=""auto""` will choose the value of the parameter automatically,based on the values of `n_samples`, `n_features`, `loss`, `multi_class`and `penalty`. If `n_samples` < `n_features` and optimizer supportschosen `loss`, `multi_class` and `penalty`, then dual will be set to True,otherwise it will be set to False... versionchanged:: 1.3 The `""auto""` option is added in version 1.3 and will be the default in version 1.5.",'auto'
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive.For an intuitive visualization of the effects of scalingthe regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1
,"multi_class multi_class: {'ovr', 'crammer_singer'}, default='ovr'Determines the multi-class strategy if `y` contains more thantwo classes.``""ovr""`` trains n_classes one-vs-rest classifiers, while``""crammer_singer""`` optimizes a joint objective over all classes.While `crammer_singer` is interesting from a theoretical perspectiveas it is consistent, it is seldom used in practice as it rarely leadsto better accuracy and is more expensive to compute.If ``""crammer_singer""`` is chosen, the options loss, penalty and dualwill be ignored.",'ovr'
,"fit_intercept fit_intercept: bool, default=TrueWhether or not to fit an intercept. If set to True, the feature vectoris extended to include an intercept term: `[x_1, ..., x_n, 1]`, where1 corresponds to the intercept. If set to False, no intercept will beused in calculations (i.e. data is expected to be already centered).",True
,"intercept_scaling intercept_scaling: float, default=1.0When `fit_intercept` is True, the instance vector x becomes ``[x_1,..., x_n, intercept_scaling]``, i.e. a ""synthetic"" feature with aconstant value equal to `intercept_scaling` is appended to the instancevector. The intercept becomes intercept_scaling * synthetic featureweight. Note that liblinear internally penalizes the intercept,treating it like any other term in the feature vector. To reduce theimpact of the regularization on the intercept, the `intercept_scaling`parameter can be set to a value greater than 1; the higher the value of`intercept_scaling`, the lower the 

In [13]:
val_scores = model.decision_function(X_val_vec)
val_scores = np.asarray(val_scores)
print("val_scores shape:", val_scores.shape)


val_scores shape: (50000, 50)


In [14]:
def top_k_binary_predictions(y_scores, k):
    y_pred = np.zeros_like(y_scores, dtype=int)
    topk_idx = np.argsort(-y_scores, axis=1)[:, :k]

    for i in range(y_scores.shape[0]):
        y_pred[i, topk_idx[i]] = 1

    return y_pred

In [15]:
def evaluate_at_k(y_true, y_scores, k):
    y_pred_k = top_k_binary_predictions(y_scores, k)

    precision = precision_score(y_true, y_pred_k, average="micro", zero_division=0)
    recall = recall_score(y_true, y_pred_k, average="micro", zero_division=0)
    f1 = f1_score(y_true, y_pred_k, average="micro", zero_division=0)

    return precision, recall, f1

In [16]:
from sklearn.metrics import precision_score, recall_score, f1_score

for k in [1, 2, 3,4]:
    p, r, f = evaluate_at_k(y_val_bin, val_scores, k)

    print(f"\n===== Top-{k} Metrics =====")
    print(f"Precision@{k}: {p:.4f}")
    print(f"Recall@{k}:    {r:.4f}")
    print(f"F1@{k}:        {f:.4f}")



===== Top-1 Metrics =====
Precision@1: 0.8451
Recall@1:    0.7423
F1@1:        0.7904

===== Top-2 Metrics =====
Precision@2: 0.5026
Recall@2:    0.8829
F1@2:        0.6406

===== Top-3 Metrics =====
Precision@3: 0.3517
Recall@3:    0.9267
F1@3:        0.5099

===== Top-4 Metrics =====
Precision@4: 0.2696
Recall@4:    0.9472
F1@4:        0.4197


In [17]:
joblib.dump(model, "best_svc2.pkl")

['best_svc2.pkl']

In [18]:
model = joblib.load("best_svc2.pkl")


In [19]:
from sklearn.metrics import precision_score, recall_score, f1_score

# 1) استخراج scores من LinearSVC على test
test_scores = model.decision_function(X_test_vec)
test_scores = np.asarray(test_scores)

In [20]:
for k in [1, 2, 3, 4]:
    p, r, f = evaluate_at_k(y_test_bin, test_scores, k)

    print(f"\n===== Top-{k} Test Metrics =====")
    print(f"Precision@{k}: {p:.4f}")
    print(f"Recall@{k}:    {r:.4f}")
    print(f"F1@{k}:        {f:.4f}")


===== Top-1 Test Metrics =====
Precision@1: 0.8442
Recall@1:    0.7410
F1@1:        0.7892

===== Top-2 Test Metrics =====
Precision@2: 0.5043
Recall@2:    0.8852
F1@2:        0.6426

===== Top-3 Test Metrics =====
Precision@3: 0.3529
Recall@3:    0.9291
F1@3:        0.5115

===== Top-4 Test Metrics =====
Precision@4: 0.2702
Recall@4:    0.9485
F1@4:        0.4205
